# 01 · Comprensión de datos — consultas de Trufi App en Cochabamba
**Fase CRISP-DM:** comprensión de los datos.

**Objetivo.** Conocer la estructura y la calidad de las fuentes, construir la variable objetivo (consultas por
zona H3 res 8) y explorar su distribución, su relación con la población y el territorio, su cobertura temporal y
su autocorrelación espacial.

**Entradas:** `data/raw/*.csv` (consultas semanales), `data/raw/gtfs/`, `data/external/kontur_population_BO_*.gpkg.gz`.

**Salidas:**
- `data/interim/`: `queries.parquet` (Hive), `queries_limpias.parquet`, `usuarios_anomalos.parquet`,
  `area_estudio.geojson`, `kontur_{2022,2023}_h3r8.parquet`, `celdas_objetivo.parquet`.
- `resultados/01_datos/`: tablas CSV, `figuras/` y `metricas.json`.

| § | Sección | Habilita |
|---|---|---|
| 1 | Entorno y regeneración | Re-ejecución limpia |
| 2 | Inventario de fuentes | Saber qué cubre cada fuente |
| 3 | Estructura | Consolidación con linaje |
| 4 | Calidad | Reglas de limpieza |
| 5 | Cobertura temporal | Tratamiento del hueco |
| 6 | Soporte espacial | Área de estudio |
| 7–8 | Variable objetivo | Construcción y distribución |
| 9 | Relaciones | Candidatas a predictores |
| 10 | Autocorrelación espacial | Validación por bloques |
| 11–12 | Diccionario y cierre | Métricas de la fase |

## 1 · Entorno y regeneración
**Objetivo:** ubicar la raíz, cargar parámetros de `config.py` y borrar las salidas previas de esta fase para
regenerarlas. **Salida:** carpetas limpias.

In [1]:
# Bootstrap: la raíz es la carpeta con pyproject.toml; se agrega a sys.path para importar config y trufi_ds
import sys
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

In [2]:
from trufi_ds.notebook_setup import *  # noqa: F403  config, pl, pd, np, plt, h3, guardar_*
from trufi_ds import eda, spatial
from trufi_ds import io as tio
from trufi_ds import preparation as prep

import geopandas as gpd
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr

FASE = "01_datos"
np.random.seed(config.SEMILLA)
M = {}  # métricas de la fase; se guardan en metricas.json al cierre

# Paleta: azul secuencial para magnitudes, gris para contexto
AZUL, NARANJA, AQUA, AMARILLO, GRIS, TINTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#c3c2b7", "#52514e"
SECUENCIAL = LinearSegmentedColormap.from_list("azul", ["#cde2fb", "#6da7ec", "#2a78d6", "#1c5cab", "#0d366b"])

In [3]:
# Borra resultados/01_datos y los intermedios que produce esta fase.
# queries.parquet solo se borra si se pide reconstruir desde los CSV (lectura lenta).
intermedios = [config.QUERIES_LIMPIAS, config.USUARIOS_ANOMALOS, config.AREA_ESTUDIO, config.KONTUR_2023_H3R8,
               config.KONTUR_2022_H3R8, config.CELDAS_OBJETIVO,
               config.DATA_INTERIM / "h3_orig.parquet", config.DATA_INTERIM / "h3_dest.parquet"]  # residuos sin lectores
if config.REGENERAR_CONSULTAS:
    intermedios.append(config.QUERIES_PARQUET)
limpiar_salidas(FASE, intermedios)
print(f"Salidas regeneradas en {config.RESULTADOS / FASE}; releer CSV: {config.REGENERAR_CONSULTAS}")

Salidas regeneradas en /home/robvox/Desktop/trufi-data-science/resultados/01_datos; releer CSV: True


## 2 · Inventario de fuentes
**Objetivo:** describir qué cubre cada fuente (período, tamaño, versión) antes de mirar su contenido.
**Salida:** `inventario_fuentes.csv`.

In [4]:
# Consultas: 85 exportaciones semanales; el período sale del nombre (<inicio>_to_<fin>_<año>-<semana>.csv)
archivos = sorted(config.DATA_RAW.glob("*.csv"))
columnas = {a.name: tio.read_csv_safe(a, n_rows=0)[0].columns for a in archivos}
grupo = eda.grupos_esquema(columnas)
inicio = min(a.name[:10] for a in archivos)
fin = max(a.name.split("_to_")[1][:10] for a in archivos)
print(f"{len(archivos)} archivos, {len(set(grupo.values()))} esquemas, período {inicio} → {fin}")

85 archivos, 2 esquemas, período 2022-09-12 → 2024-06-09


In [5]:
# Kontur Population (H3 res 8): dos ediciones; la de 2023 es la exposición, la de 2022 sirve de contraste
kontur23 = prep.load_kontur(config.KONTUR_2023_GPKG_GZ).sort("h3_cell")
kontur22 = prep.load_kontur(config.KONTUR_2022_GPKG_GZ).sort("h3_cell")
assert {h3.get_resolution(c) for c in kontur23["h3_cell"].head(1000)} == {config.H3_RES}, "Kontur no está en res 8"
kontur23.write_parquet(config.KONTUR_2023_H3R8)
kontur22.write_parquet(config.KONTUR_2022_H3R8)
for nombre, k in [("2023", kontur23), ("2022", kontur22)]:
    print(f"Kontur {nombre}: {k.height:,} celdas, {k['population'].sum():,} habitantes (Bolivia)")

Kontur 2023: 145,929 celdas, 12,431,735 habitantes (Bolivia)
Kontur 2022: 178,201 celdas, 9,995,771 habitantes (Bolivia)


In [6]:
# GTFS: tamaño del feed y vigencia declarada (feed_info) frente al período de las consultas
gtfs = {n: pd.read_csv(config.GTFS_DIR / f"{n}.txt", dtype=str) for n in ["agency", "routes", "trips", "stops", "shapes"]}
feed = pd.read_csv(config.GTFS_DIR / "feed_info.txt", dtype=str).iloc[0]
vigencia = f"{feed['feed_start_date']} → {feed['feed_end_date']}"
print(f"GTFS: {len(gtfs['routes'])} líneas, {len(gtfs['trips'])} recorridos, {gtfs['shapes']['shape_id'].nunique()} trazados, "
      f"{len(gtfs['stops']):,} paradas; vigencia {vigencia}")

GTFS: 141 líneas, 626 recorridos, 626 trazados, 22,320 paradas; vigencia 20240101 → 20261231


In [7]:
inventario = pl.DataFrame([
    {"fuente": "consultas Trufi App", "detalle": f"{len(archivos)} CSV semanales", "periodo": f"{inicio} → {fin}",
     "unidad": "consulta de ruta"},
    {"fuente": "Kontur Population 2023-11-01", "detalle": f"{kontur23.height:,} celdas", "periodo": "estimación 2023",
     "unidad": "celda H3 res 8"},
    {"fuente": "Kontur Population 2022-06-30", "detalle": f"{kontur22.height:,} celdas", "periodo": "estimación 2022",
     "unidad": "celda H3 res 8"},
    {"fuente": "GTFS Trufi Cochabamba", "detalle": f"{len(gtfs['routes'])} líneas, {len(gtfs['stops']):,} paradas",
     "periodo": vigencia, "unidad": "trazado y parada"},
])
M.update({"csv_archivos": len(archivos), "csv_esquemas": len(set(grupo.values())), "periodo_inicio": inicio,
          "periodo_fin": fin, "kontur_2023_celdas": kontur23.height, "kontur_2022_celdas": kontur22.height,
          "gtfs_lineas": len(gtfs["routes"]), "gtfs_paradas": len(gtfs["stops"]), "gtfs_vigencia": vigencia})
guardar_tabla(inventario, "inventario_fuentes", FASE)

fuente,detalle,periodo,unidad
str,str,str,str
"""consultas Trufi App""","""85 CSV semanales""","""2022-09-12 → 2024-06-09""","""consulta de ruta"""
"""Kontur Population 2023-11-01""","""145,929 celdas""","""estimación 2023""","""celda H3 res 8"""
"""Kontur Population 2022-06-30""","""178,201 celdas""","""estimación 2022""","""celda H3 res 8"""
"""GTFS Trufi Cochabamba""","""141 líneas, 22,320 paradas""","""20240101 → 20261231""","""trazado y parada"""


## 3 · Estructura
**Objetivo:** consolidar los CSV en una sola tabla con linaje (`source_file`, `source_batch`, `source_encoding`) y
columnas homogéneas. **Salida:** `data/interim/queries.parquet` (particionado por año/semana ISO) y
`archivos_consultas.csv`.

In [8]:
# Dos esquemas: el lote de 2024 trae coordenadas con nombre largo y columnas extra; se renombran a un solo esquema.
# Algunos archivos vienen en Latin-1: read_csv_safe los relee sin perder acentos.
if config.REGENERAR_CONSULTAS or not config.QUERIES_PARQUET.exists():
    crudas, _ = tio.leer_consultas(archivos)
    n_escritas = tio.escribir_hive(crudas, config.QUERIES_PARQUET, ["year", "week"])
    assert n_escritas == crudas.height, "El Parquet no conserva todas las filas"
    del crudas
# Siempre se trabaja sobre lo escrito, en orden estable, para que ambas rutas den el mismo resultado
consultas = pl.read_parquet(config.QUERIES_PARQUET).with_columns(pl.col("year", "week").cast(pl.Int32))
consultas = consultas.sort(["source_file", "ts", "userID", "lat_orig", "lon_orig", "lat_dest", "lon_dest"])
N_CRUDAS = consultas.height
print(f"Consultas consolidadas: {N_CRUDAS:,} filas × {consultas.width} columnas")

Consultas consolidadas: 1,927,675 filas × 21 columnas


In [9]:
# Registro por archivo: lote, codificación, grupo de esquema y filas
archivos_tab = (consultas.group_by("source_file", "source_batch", "source_encoding").len()
                .rename({"source_file": "archivo", "source_batch": "lote", "source_encoding": "codificacion",
                         "len": "filas"})
                .with_columns(pl.col("archivo").replace_strict(grupo, return_dtype=pl.Int64).alias("grupo_esquema"))
                .sort("archivo"))
assert archivos_tab.height == len(archivos), "Algún CSV no aportó filas"
M.update({"consultas_crudas": N_CRUDAS,
          "csv_latin1": archivos_tab.filter(pl.col("codificacion") == "latin-1").height})
print(archivos_tab.group_by("lote", "codificacion", "grupo_esquema").agg(pl.len().alias("archivos"), pl.col("filas").sum()))
guardar_tabla(archivos_tab, "archivos_consultas", FASE).head()

shape: (2, 5)
┌─────────────────┬──────────────┬───────────────┬──────────┬─────────┐
│ lote            ┆ codificacion ┆ grupo_esquema ┆ archivos ┆ filas   │
│ ---             ┆ ---          ┆ ---           ┆ ---      ┆ ---     │
│ str             ┆ str          ┆ i64           ┆ u32      ┆ u32     │
╞═════════════════╪══════════════╪═══════════════╪══════════╪═════════╡
│ lote_2024_nuevo ┆ latin-1      ┆ 1             ┆ 6        ┆ 161777  │
│ lote_original   ┆ utf-8        ┆ 0             ┆ 79       ┆ 1765898 │
└─────────────────┴──────────────┴───────────────┴──────────┴─────────┘


archivo,lote,codificacion,filas,grupo_esquema
str,str,str,u32,i64
"""2022-09-12_to_2022-09-18_2022-…","""lote_original""","""utf-8""",5049,0
"""2022-09-19_to_2022-09-25_2022-…","""lote_original""","""utf-8""",10838,0
"""2022-09-26_to_2022-10-02_2022-…","""lote_original""","""utf-8""",10850,0
"""2022-10-03_to_2022-10-09_2022-…","""lote_original""","""utf-8""",11438,0
"""2022-10-10_to_2022-10-16_2022-…","""lote_original""","""utf-8""",11443,0


## 4 · Calidad
**Objetivo:** medir nulos, duplicados, coordenadas imposibles, la unidad y el rango de `distancia` y el uso
anómalo por usuario. Cada problema se mide sobre las filas crudas (los conteos pueden solaparse); el orden en que
se eliminan se fija en la §7. **Salidas:** `nulos.csv`, `duplicados.csv`, `distancia.csv`, `usuarios_senales.csv`,
`calidad_resumen.csv`, `data/interim/usuarios_anomalos.parquet`.

In [10]:
# Nulos por columna y por lote: las columnas exclusivas de un lote son nulos estructurales, no pérdida de datos
nulos = eda.perfil_nulos(consultas)
guardar_tabla(nulos, "nulos", FASE).filter(pl.col("pct_global") > 0)

columna,pct_global,lote_2024_nuevo,lote_original
str,f64,f64,f64
"""time_of_day""",91.608,0.0,100.0
"""year_week_number""",91.608,0.0,100.0


In [11]:
# Cuatro definiciones de duplicado: exacto (todas las columnas de datos) y variantes por OD, tiempo y usuario
datos_cols = [c for c in consultas.columns if c not in tio.LINAJE]
definiciones = {
    "exacto": datos_cols,
    "od_ts": ["lat_orig", "lon_orig", "lat_dest", "lon_dest", "ts"],
    "usuario_ts": ["userID", "ts"],
    "od_ts_usuario": ["lat_orig", "lon_orig", "lat_dest", "lon_dest", "ts", "userID"],
}
dups = eda.duplicados(consultas, definiciones)
M["duplicados_exactos_copias"] = int(dups.filter(pl.col("definicion") == "exacto")["copias_sobrantes"].item())
guardar_tabla(dups, "duplicados", FASE)

definicion,columnas,filas_implicadas,copias_sobrantes,pct_copias
str,str,i64,i64,f64
"""exacto""","""date, lat_orig, lon_orig, lat_…",104,60,0.0031
"""od_ts""","""lat_orig, lon_orig, lat_dest, …",104,60,0.0031
"""usuario_ts""","""userID, ts""",137,90,0.0047
"""od_ts_usuario""","""lat_orig, lon_orig, lat_dest, …",104,60,0.0031


In [12]:
# Coordenadas: (0,0) es un GPS sin señal; fuera del rectángulo de Bolivia es físicamente imposible para esta app
lat, lon = pl.col("lat_orig"), pl.col("lon_orig")
cero = (lat == 0) | (lon == 0)
BB = config.BBOX_BOLIVIA
en_bolivia = lat.is_between(BB["lat_min"], BB["lat_max"]) & lon.is_between(BB["lon_min"], BB["lon_max"])
coords_ok = ~cero & en_bolivia
M.update({"origen_cero": consultas.filter(cero).height,
          "origen_fuera_bolivia": consultas.filter(~cero & ~en_bolivia).height})
print({k: M[k] for k in ("origen_cero", "origen_fuera_bolivia")})

{'origen_cero': 3, 'origen_fuera_bolivia': 9}


In [13]:
# distancia: se contrasta con la geodésica recalculada para confirmar la unidad; luego percentiles y colas
M.update(eda.verificar_unidad_distancia(consultas, 200_000, config.SEMILLA))
d_km = consultas.filter(coords_ok & (pl.col("distancia") > 0))["distancia"].to_numpy() / 1000
filas = [{"medida": f"P{p}", "km": round(float(np.percentile(d_km, p)), 2)} for p in (50, 90, 99, 99.9)]
filas += [{"medida": f"pct_mayor_{u}km", "km": round(float((d_km > u).mean() * 100), 3)} for u in (20, 30, 40, 50)]
distancia = pl.DataFrame(filas)
M.update({"distancia_p99_9_km": filas[3]["km"],
          "distancia_pct_mayor_umbral": round(float((d_km > config.DIST_MAX_M / 1000).mean() * 100), 3)})
print({k: v for k, v in M.items() if k.startswith("distancia")})
guardar_tabla(distancia, "distancia", FASE)

{'distancia_unidad': 'metros', 'distancia_error_mediano_pct': 0.142, 'distancia_correlacion_geodesica': 1.0, 'distancia_p99_9_km': 40.7, 'distancia_pct_mayor_umbral': 0.04}


medida,km
str,f64
"""P50""",3.39
"""P90""",10.26
"""P99""",19.69
"""P99.9""",40.7
"""pct_mayor_20km""",0.961
"""pct_mayor_30km""",0.353
"""pct_mayor_40km""",0.109
"""pct_mayor_50km""",0.04


In [14]:
# El umbral de distancia: la cola > 40 km ya no es movilidad dentro del eje metropolitano (≈ P99,9);
# 50 km conserva los viajes del valle alto hacia la ciudad, que son largos pero metropolitanos
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(d_km, bins=np.logspace(-1, np.log10(d_km.max()), 80), color=AZUL, edgecolor="white", linewidth=0.3)
ax.axvline(config.DIST_MAX_M / 1000, color=TINTA, lw=1.5, ls="--", label=f"umbral {config.DIST_MAX_M // 1000} km")
ax.axvline(np.percentile(d_km, 99.9), color=GRIS, lw=1.5, label=f"P99,9 = {np.percentile(d_km, 99.9):.1f} km")
ax.set(xscale="log", yscale="log", xlabel="distancia origen–destino (km)", ylabel="consultas",
       title=f"Distribución de la distancia por consulta (N = {d_km.size:,})")
ax.legend(frameon=False)
guardar_figura(fig, "distancia", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/distancia.png')

In [15]:
# Uso anómalo: seis señales por usuario (volumen, intensidad, dispersión, ráfaga, sin rutina, rapidez);
# se marca con al menos MIN_SENALES señales para no castigar a usuarios intensivos legítimos
validas = consultas.filter(coords_ok & (pl.col("lat_dest") != 0) & pl.col("lat_dest").is_between(-90, 90)
                           & pl.col("lon_dest").is_between(-180, 180))
validas = prep.add_cells(prep.add_cells(validas, "lat_orig", "lon_orig", "h3_origin"),
                         "lat_dest", "lon_dest", "h3_destination")
umbrales = {"volumen": config.SENAL_VOLUMEN, "intensidad": config.SENAL_INTENSIDAD,
            "dispersion": config.SENAL_DISPERSION, "rafaga": config.SENAL_RAFAGA,
            "sin_rutina": config.SENAL_SIN_RUTINA, "rapido_seg": config.SENAL_RAPIDO_SEG}
usuarios = eda.senales_usuario(validas, umbrales, config.MIN_SENALES)
anomalos = usuarios.filter(pl.col("anomalo")).select("userID", "n_consultas", "n_senales")
anomalos.write_parquet(config.USUARIOS_ANOMALOS)
del validas

In [16]:
senales = pl.DataFrame([{"senal": s, "usuarios": int(usuarios[s].sum())} for s in usuarios.columns if s.startswith("s_")])
M.update({"usuarios": usuarios.height, "usuarios_anomalos": anomalos.height,
          "usuarios_mediana_consultas": float(usuarios["n_consultas"].median()),
          "consultas_de_anomalos": int(anomalos["n_consultas"].sum())})
print({k: M[k] for k in ("usuarios", "usuarios_anomalos", "usuarios_mediana_consultas", "consultas_de_anomalos")})
guardar_tabla(senales, "usuarios_senales", FASE)

{'usuarios': 130545, 'usuarios_anomalos': 2, 'usuarios_mediana_consultas': 5.0, 'consultas_de_anomalos': 233}


senal,usuarios
str,i64
"""s_volumen""",1
"""s_intensidad""",1
"""s_dispersion""",0
"""s_rafaga""",1
"""s_sin_rutina""",0
"""s_rapido""",10511


In [17]:
# Resumen de calidad sobre filas crudas (no excluyentes entre sí)
calidad = pl.DataFrame([
    {"problema": "origen (0,0)", "filas": M["origen_cero"]},
    {"problema": "origen fuera de Bolivia", "filas": M["origen_fuera_bolivia"]},
    {"problema": "copia de duplicado exacto", "filas": M["duplicados_exactos_copias"]},
    {"problema": f"distancia > {config.DIST_MAX_M // 1000} km", "filas": consultas.filter(pl.col("distancia") > config.DIST_MAX_M).height},
    {"problema": "consulta de usuario anómalo", "filas": M["consultas_de_anomalos"]},
]).with_columns((pl.col("filas") / N_CRUDAS * 100).round(3).alias("pct"))
guardar_tabla(calidad, "calidad_resumen", FASE)

problema,filas,pct
str,i64,f64
"""origen (0,0)""",3,0.0
"""origen fuera de Bolivia""",9,0.0
"""copia de duplicado exacto""",60,0.003
"""distancia > 50 km""",778,0.04
"""consulta de usuario anómalo""",233,0.012


## 5 · Cobertura temporal
**Objetivo:** ver qué semanas tienen datos, dónde está el hueco y si el volumen cambia antes y después de él.
Se cuentan filas sin copias duplicadas (dos archivos repiten la última semana de 2022) y se distinguen
semanas completas (7 días con datos) de parciales (exportaciones cortadas).
**Salida:** `cobertura_semanal.csv`, figura `cobertura_semanal.png`.

In [18]:
unicas = consultas.filter(pl.struct(datos_cols).is_first_distinct())
semanal = eda.cobertura_semanal(unicas)
faltan = semanal.filter(~pl.col("observada"))
parciales = semanal.filter(pl.col("observada") & ~pl.col("completa"))
# Hueco = semanas faltantes consecutivas; la media semanal se compara solo con semanas completas
i0, i1 = semanal["observada"].arg_min(), semanal.with_row_index().filter(~pl.col("observada"))["index"].max()
media = lambda s: round(s.filter(pl.col("completa"))["consultas"].mean())
M.update({"semanas_calendario": semanal.height, "semanas_observadas": int(semanal["observada"].sum()),
          "semanas_completas": int(semanal["completa"].sum()), "semanas_parciales": parciales.height,
          "semanas_faltantes": faltan.height, "hueco_desde": str(faltan["inicio"].min()),
          "hueco_hasta": str(faltan["inicio"].max()), "media_semanal_antes_hueco": media(semanal[:i0]),
          "media_semanal_despues_hueco": media(semanal[i1 + 1:])})
print(parciales.select("inicio", "consultas", "dias_con_datos"))
guardar_tabla(semanal, "cobertura_semanal", FASE).tail(3)

shape: (6, 3)
┌────────────┬───────────┬────────────────┐
│ inicio     ┆ consultas ┆ dias_con_datos │
│ ---        ┆ ---       ┆ ---            │
│ date       ┆ u32       ┆ u32            │
╞════════════╪═══════════╪════════════════╡
│ 2023-07-24 ┆ 13311     ┆ 5              │
│ 2024-01-29 ┆ 20340     ┆ 5              │
│ 2024-02-05 ┆ 16224     ┆ 4              │
│ 2024-03-04 ┆ 33843     ┆ 6              │
│ 2024-04-29 ┆ 932       ┆ 1              │
│ 2024-06-03 ┆ 1386      ┆ 1              │
└────────────┴───────────┴────────────────┘


year,week,inicio,consultas,dias_con_datos,observada,completa
i32,i32,date,u32,u32,bool,bool
2024,21,2024-05-20,41508,7,true,true
2024,22,2024-05-27,33872,7,true,true
2024,23,2024-06-03,1386,1,true,false


In [19]:
# Las semanas sin datos se dejan vacías (no en cero) para no dibujar una caída que no existe
serie = semanal.with_columns(pl.when(pl.col("observada")).then(pl.col("consultas")).alias("y"))
fig, ax = plt.subplots(figsize=(11, 3.8))
ax.plot(serie["inicio"], serie["y"].cast(pl.Float64).fill_null(np.nan), color=AZUL, lw=2)
ax.scatter(parciales["inicio"], parciales["consultas"], s=40, facecolor="white", edgecolor=TINTA, zorder=3,
           label=f"{parciales.height} semanas parciales (< 7 días)")
ax.axvspan(faltan["inicio"].min(), faltan["inicio"].max(), color=GRIS, alpha=0.4, lw=0,
           label=f"{faltan.height} semanas sin datos")
ax.set(ylabel="consultas por semana", title="Consultas semanales (sin copias duplicadas)")
ax.legend(frameon=False, loc="upper left")
guardar_figura(fig, "cobertura_semanal", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/cobertura_semanal.png')

## 6 · Soporte espacial
**Objetivo:** delimitar dónde se mide la demanda a partir de los propios datos. Se toman los orígenes válidos
(coordenadas correctas, usuario no anómalo, **sin** filtro de distancia), la componente principal de celdas H3
ocupadas y contiguas, y su envolvente convexa con un margen. **Salida:** `area_variantes.csv`,
`data/interim/area_estudio.geojson`, figura `area_estudio.png`.

In [20]:
# Orígenes que definen el área: solo ubicación (no conteos); la distancia filtra consultas, no dibuja el área
es_anomalo = pl.col("userID").is_in(anomalos["userID"].implode())
origenes = prep.add_cells(consultas.filter(coords_ok & ~es_anomalo).select("lat_orig", "lon_orig"),
                          "lat_orig", "lon_orig", "h3_origin")
ocupadas = set(origenes["h3_origin"].unique().to_list())
lat_o, lon_o = origenes["lat_orig"].to_numpy(), origenes["lon_orig"].to_numpy()
print(f"Orígenes válidos: {origenes.height:,} en {len(ocupadas):,} celdas H3 ocupadas")

Orígenes válidos: 1,927,430 en 1,516 celdas H3 ocupadas


In [21]:
# Variantes de la regla: sin componente la envolvente se estira hasta orígenes aislados en otras ciudades;
# con k > 1 se ganan cientos de km² casi vacíos por muy pocas consultas
variantes, geoms = [], {}
for k in [None, 1, 2, 3]:
    celdas_k = ocupadas if k is None else prep.main_component(ocupadas, k)
    pts = origenes.filter(pl.col("h3_origin").is_in(list(celdas_k))).unique(["lat_orig", "lon_orig"])
    geoms[k], km2 = prep.hull_area(pts["lat_orig"].to_numpy(), pts["lon_orig"].to_numpy(), config.BUFFER_AREA_M)
    dentro = prep.points_in(geoms[k], lat_o, lon_o)
    variantes.append({"regla": "todas las celdas" if k is None else f"componente k={k}", "celdas": len(celdas_k),
                      "area_km2": round(km2, 1), "pct_origenes_dentro": round(float(dentro.mean() * 100), 3)})
guardar_tabla(pl.DataFrame(variantes), "area_variantes", FASE)

regla,celdas,area_km2,pct_origenes_dentro
str,i64,f64,f64
"""todas las celdas""",1516,485940.1,100.0
"""componente k=1""",919,1505.7,99.87
"""componente k=2""",1076,1965.4,99.933
"""componente k=3""",1195,3040.9,99.962


In [22]:
AREA = geoms[config.K_COMPONENTE]
gpd.GeoDataFrame({"regla": [f"componente k={config.K_COMPONENTE} + {config.BUFFER_AREA_M} m"]},
                 geometry=[AREA], crs="EPSG:4326").to_file(config.AREA_ESTUDIO, driver="GeoJSON")
CENTRO = prep.area_center(AREA)
dentro = prep.points_in(AREA, lat_o, lon_o)
M.update({"area_km2": round(prep.area_km2(AREA), 1), "area_pct_origenes_dentro": round(float(dentro.mean() * 100), 3),
          "area_centro_a_plaza_km": round(float(prep.haversine_km(np.array([CENTRO[0]]), np.array([CENTRO[1]]),
                                                                   config.PLAZA_14_SEPTIEMBRE["lat"],
                                                                   config.PLAZA_14_SEPTIEMBRE["lon"])[0]), 2)})
print({k: M[k] for k in ("area_km2", "area_pct_origenes_dentro", "area_centro_a_plaza_km")})

{'area_km2': 1505.7, 'area_pct_origenes_dentro': 99.87, 'area_centro_a_plaza_km': 7.82}


In [23]:
muestra = np.random.default_rng(config.SEMILLA).choice(lat_o.size, 60_000, replace=False)
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(lon_o[muestra], lat_o[muestra], s=1, color=AZUL, alpha=0.08, lw=0)
gpd.GeoSeries([AREA], crs="EPSG:4326").boundary.plot(ax=ax, color=TINTA, lw=1.5)
ax.plot(*CENTRO[::-1], marker="*", color=NARANJA, ms=14, ls="", label="centroide del área")
ax.plot(config.PLAZA_14_SEPTIEMBRE["lon"], config.PLAZA_14_SEPTIEMBRE["lat"], marker="^", color=TINTA, ms=8, ls="",
        label="Plaza 14 de Septiembre")
minx, miny, maxx, maxy = AREA.bounds
ax.set(xlim=(minx - 0.05, maxx + 0.05), ylim=(miny - 0.05, maxy + 0.05), xlabel="longitud", ylabel="latitud",
       title=f"Área de estudio: {M['area_km2']:,} km², {M['area_pct_origenes_dentro']} % de los orígenes válidos")
ax.legend(frameon=False, loc="upper right")
guardar_figura(fig, "area_estudio", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/area_estudio.png')

## 7 · Variable objetivo: construcción
**Objetivo:** construir `query_count`, el número de consultas con origen en cada celda H3 res 8 del área.

Pasos, en este orden (cada uno se mide en `flujo_limpieza.csv`):
1. Quitar orígenes en (0,0) y fuera de Bolivia.
2. Quitar copias de duplicados exactos (se conserva una).
3. Quitar orígenes fuera del área de estudio.
4. Quitar consultas con `distancia` mayor que `DIST_MAX_M`.
5. Quitar consultas de usuarios anómalos.
6. Asignar la celda H3 del origen y contar consultas (y usuarios distintos) por celda.
7. Unir las celdas Kontur del área: las celdas pobladas sin consultas entran con 0; las celdas con consultas sin
   registro Kontur quedan con población 0 (ausencia estructural, no imputación).

**Salidas:** `flujo_limpieza.csv`, `data/interim/queries_limpias.parquet`, `data/interim/celdas_objetivo.parquet`.

In [24]:
limpias, flujo = prep.clean_queries(consultas, AREA, anomalos["userID"], config.DIST_MAX_M)
limpias = limpias.sort(["source_file", "ts", "userID", "lat_origin", "lon_origin"])
limpias.write_parquet(config.QUERIES_LIMPIAS)
flujo = flujo.with_columns((pl.col("filas_despues") / N_CRUDAS * 100).round(3).alias("pct_conservado"))
M.update({"consultas_validas": limpias.height, "pct_consultas_validas": round(limpias.height / N_CRUDAS * 100, 2)})
guardar_tabla(flujo, "flujo_limpieza", FASE)

regla,filas_eliminadas,filas_despues,pct_conservado
str,i64,i64,f64
"""consultas crudas""",0,1927675,100.0
"""origen (0,0)""",3,1927672,100.0
"""origen fuera de Bolivia""",9,1927663,99.999
"""copia de duplicado exacto""",60,1927603,99.996
"""origen fuera del área de estud…",2509,1925094,99.866
"""distancia > 50 km""",283,1924811,99.851
"""usuario anómalo""",233,1924578,99.839


In [25]:
celdas = (prep.build_cell_table(limpias, kontur23, AREA)
          .join(kontur22.rename({"population": "population_2022"}), on="h3_cell", how="left")
          .with_columns(pl.col("population_2022").fill_null(0)))
lat_c, lon_c = prep.cell_centroids(celdas["h3_cell"].to_list())
celdas = celdas.with_columns(pl.Series("lat", lat_c), pl.Series("lon", lon_c)).sort("h3_cell")
celdas.write_parquet(config.CELDAS_OBJETIVO)
assert celdas["query_count"].sum() == limpias.height, "Se perdieron consultas al agregar por celda"
assert celdas["h3_cell"].is_unique().all() and celdas.null_count().sum_horizontal().item() == 0
print(f"Celdas: {celdas.height:,}  |  columnas: {celdas.columns}")

Celdas: 1,628  |  columnas: ['h3_cell', 'query_count', 'user_count', 'population', 'population_2022', 'lat', 'lon']


## 8 · Variable objetivo: análisis
**Objetivo:** describir `query_count` y la tasa por habitante: ceros, sobredispersión, concentración y geografía.
La tasa se calcula solo en celdas con `population ≥ POP_MIN`, porque con pocos habitantes es inestable.
**Salidas:** `objetivo_resumen.csv`, figuras `objetivo_distribucion.png` y `tasa_mapa.png`.

In [26]:
q, pob = celdas["query_count"].to_numpy(), celdas["population"].to_numpy()
modelo = celdas.filter(pl.col("population") >= config.POP_MIN).with_columns(
    (pl.col("query_count") / pl.col("population") * 1000).alias("tasa_1000"))
qm = modelo["query_count"].to_numpy()
resumen = {
    "celdas_area": celdas.height, "celdas_con_consultas": int((q > 0).sum()),
    "celdas_pobladas_sin_consultas": int(((pob > 0) & (q == 0)).sum()),
    "celdas_poblacion_min": modelo.height, "pct_ceros_poblacion_min": round(float((qm == 0).mean() * 100), 2),
    "consultas_en_celdas_poblacion_min": int(qm.sum()), "media_query_count": round(float(qm.mean()), 1),
    "varianza_sobre_media": round(float(qm.var() / qm.mean()), 1), "gini_query_count": round(prep.gini(qm), 3),
    "pct_consultas_top10_celdas": round(float(np.sort(qm)[::-1][:10].sum() / qm.sum() * 100), 1),
    "tasa_global_1000": round(float(qm.sum() / modelo["population"].sum() * 1000), 1),
    "tasa_mediana_1000": round(float(modelo["tasa_1000"].median()), 1),
    "poblacion_area": int(pob.sum()),
}
M.update(resumen)
guardar_tabla(pl.DataFrame({"medida": list(resumen), "valor": [float(v) for v in resumen.values()]}), "objetivo_resumen", FASE)

medida,valor
str,f64
"""celdas_area""",1628.0
"""celdas_con_consultas""",1017.0
"""celdas_pobladas_sin_consultas""",611.0
"""celdas_poblacion_min""",1381.0
"""pct_ceros_poblacion_min""",32.15
…,…
"""gini_query_count""",0.948
"""pct_consultas_top10_celdas""",42.0
"""tasa_global_1000""",1570.0


In [27]:
# Izquierda: distribución (log) con los ceros aparte; derecha: curva de Lorenz de la concentración
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
a1.hist(np.log10(qm[qm > 0]), bins=50, color=AZUL, edgecolor="white", linewidth=0.3)
a1.set(xlabel="log10(query_count), celdas con consultas", ylabel="celdas",
       title=f"query_count: {resumen['pct_ceros_poblacion_min']} % de celdas en cero")
lx, ly = eda.lorenz(qm)
a2.plot(lx * 100, ly * 100, color=AZUL, lw=2)
a2.plot([0, 100], [0, 100], color=GRIS, lw=1, ls="--")
a2.set(xlabel="% de celdas (orden creciente)", ylabel="% de consultas acumuladas",
       title=f"Concentración: Gini = {resumen['gini_query_count']}")
guardar_figura(fig, "objetivo_distribucion", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/objetivo_distribucion.png')

In [28]:
g = gpd.GeoDataFrame(modelo.to_pandas(), geometry=prep.cell_polygons(modelo["h3_cell"].to_list()), crs="EPSG:4326")
fig, ax = plt.subplots(figsize=(9, 8))
g.assign(log_tasa=np.log10(g["tasa_1000"] + 1)).plot(
    ax=ax, column="log_tasa", cmap=SECUENCIAL, edgecolor="none", legend=True,
    legend_kwds={"label": "log10(consultas por 1.000 hab. + 1)", "shrink": 0.7})
gpd.GeoSeries([AREA], crs="EPSG:4326").boundary.plot(ax=ax, color=TINTA, lw=1)
ax.set(xlabel="longitud", ylabel="latitud", title=f"Tasa de consultas por celda (population ≥ {config.POP_MIN})")
guardar_figura(fig, "tasa_mapa", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/tasa_mapa.png')

## 9 · Relaciones entre variables
**Objetivo:** ver con qué se asocian el conteo y la tasa: población de la celda y de sus coronas H3, distancia al
centro (centroide del área y Plaza como referencia), distancia al trazado GTFS y municipio. Spearman, porque las
relaciones son monótonas pero no lineales. Son asociaciones exploratorias sobre todas las celdas, no efectos.
**Salidas:** `relaciones.csv`, `objetivo_por_municipio.csv`, figura `objetivo_vs_poblacion.png`.

In [29]:
# Variables de contexto solo para explorar; la preparación decide cuáles entran al modelo
ctx = prep.add_territorial_features(modelo, kontur23, AREA, CENTRO)
ctx = prep.add_municipality(ctx, limpias).with_columns(
    pl.Series("dist_plaza_km", prep.haversine_km(ctx["lat"].to_numpy(), ctx["lon"].to_numpy(),
                                                 config.PLAZA_14_SEPTIEMBRE["lat"], config.PLAZA_14_SEPTIEMBRE["lon"])),
    pl.Series("dist_trazado_m", spatial.distancia_trazado_m(ctx["lat"].to_numpy(), ctx["lon"].to_numpy(),
                                                            config.GTFS_DIR, config.MUESTREO_TRAZADO_M)))
ctx = ctx.with_columns((pl.col("dist_trazado_m") <= config.COBERTURA_M).alias("cubierta"))
print(f"Celdas cubiertas (≤ {config.COBERTURA_M} m del trazado): {ctx['cubierta'].sum():,} de {ctx.height:,}")

Celdas cubiertas (≤ 500 m del trazado): 626 de 1,381


In [30]:
variables = ["population", "pop_ring1", "pop_ring2", "dist_centro_km", "dist_plaza_km", "dist_trazado_m"]
filas = []
for v in variables:
    for obj in ["query_count", "tasa_1000"]:
        rho, p = spearmanr(ctx[v].to_numpy(), ctx[obj].to_numpy())
        filas.append({"variable": v, "objetivo": obj, "spearman": round(float(rho), 3), "p_valor": float(p)})
rho_k, p_k = spearmanr(ctx["population"].to_numpy(), ctx["population_2022"].to_numpy())
filas.append({"variable": "population_2022", "objetivo": "population", "spearman": round(float(rho_k), 3), "p_valor": float(p_k)})
relaciones = pl.DataFrame(filas)
M.update({f"spearman_{r['objetivo']}_{r['variable']}": r["spearman"] for r in filas})
guardar_tabla(relaciones, "relaciones", FASE)

variable,objetivo,spearman,p_valor
str,str,f64,f64
"""population""","""query_count""",0.855,0.0
"""population""","""tasa_1000""",0.71,1.2482e-212
"""pop_ring1""","""query_count""",0.836,0.0
"""pop_ring1""","""tasa_1000""",0.721,3.2992e-222
"""pop_ring2""","""query_count""",0.804,0.0
…,…,…,…
"""dist_plaza_km""","""query_count""",-0.697,4.8867e-201
"""dist_plaza_km""","""tasa_1000""",-0.677,1.6079e-185
"""dist_trazado_m""","""query_count""",-0.763,2.0772e-263


In [31]:
# Contraste por cobertura: tasa mediana en celdas cerca y lejos del trazado (asociación descriptiva)
cob = ctx.group_by("cubierta").agg(pl.len().alias("celdas"), pl.col("tasa_1000").median().alias("tasa_mediana"),
                                   pl.col("query_count").sum().alias("consultas")).sort("cubierta")
M.update({"celdas_cubiertas": int(ctx["cubierta"].sum()),
          "pct_consultas_celdas_cubiertas": round(float(cob.filter(pl.col("cubierta"))["consultas"].item()
                                                        / cob["consultas"].sum() * 100), 1),
          "tasa_mediana_cubiertas": round(float(cob.filter(pl.col("cubierta"))["tasa_mediana"].item()), 1),
          "tasa_mediana_no_cubiertas": round(float(cob.filter(~pl.col("cubierta"))["tasa_mediana"].item()), 1)})
cob

cubierta,celdas,tasa_mediana,consultas
bool,u32,f64,i64
false,755,0.0,13038
true,626,117.952749,1911298


In [32]:
por_mun = (ctx.group_by("municipality").agg(pl.len().alias("celdas"), pl.col("population").sum(),
                                            pl.col("query_count").sum())
           .with_columns((pl.col("query_count") / pl.col("population") * 1000).round(1).alias("tasa_1000"))
           .sort(["query_count", "municipality"], descending=[True, False]))
M["municipios_en_area"] = por_mun.height
guardar_tabla(por_mun, "objetivo_por_municipio", FASE).head(12)

municipality,celdas,population,query_count,tasa_1000
str,u32,i64,i64,f64
"""Cochabamba""",318,622120,1662310,2672.0
"""Sacaba""",259,180427,90113,499.4
"""Quillacollo""",137,146291,65275,446.2
"""Colcapirhua""",35,47652,53209,1116.6
"""Tiquipaya""",47,60991,46785,767.1
…,…,…,…,…
"""Arbieto""",86,17368,546,31.4
"""Villa Punata""",74,23430,488,20.8
"""Tolata""",37,5115,302,59.0


In [33]:
fig, ax = plt.subplots(figsize=(8, 6))
for cub, color, etiqueta in [(False, GRIS, f"> {config.COBERTURA_M} m del trazado"),
                             (True, AZUL, f"≤ {config.COBERTURA_M} m del trazado")]:
    s = ctx.filter(pl.col("cubierta") == cub)
    ax.scatter(s["population"], s["query_count"] + 1, s=10, color=color, alpha=0.6, lw=0, label=etiqueta)
ax.set(xscale="log", yscale="log", xlabel="población de la celda (Kontur 2023)", ylabel="query_count + 1",
       title=f"Consultas frente a población (Spearman = {M['spearman_query_count_population']})")
ax.legend(frameon=False)
guardar_figura(fig, "objetivo_vs_poblacion", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/objetivo_vs_poblacion.png')

## 10 · Autocorrelación espacial
**Objetivo:** medir si celdas vecinas se parecen (I de Moran) y a qué distancia deja de ocurrir (correlograma por
anillo H3), y localizar clusters (LISA). Se usa `log(1 + tasa)` para que unas pocas celdas extremas no dominen.
Si la autocorrelación es alta, la validación del modelo tiene que ser por bloques espaciales.
**Salidas:** `autocorrelacion.csv`, figura `lisa.png`.

In [34]:
celdas_m = ctx["h3_cell"].to_list()
y_tasa = np.log1p(ctx["tasa_1000"].to_numpy())
corr = eda.correlograma_moran(y_tasa, celdas_m, config.K_MAX_CORRELOGRAMA, config.PERMUTACIONES, config.SEMILLA)
# Arista media H3 res 8 ≈ 0,53 km y apotema ≈ 0,46 km: el anillo k está a ≈ 0,92·k km del centro
corr = corr.with_columns((pl.col("orden_k") * 0.92).round(2).alias("distancia_aprox_km"))
M.update({"moran_I_k1": corr["moran_I"][0], "moran_p_k1": corr["p_sim"][0],
          "moran_I_k_max": corr["moran_I"][-1]})
guardar_tabla(corr, "autocorrelacion", FASE)

orden_k,moran_I,p_sim,distancia_aprox_km
i64,f64,f64,f64
1,0.7161,0.001,0.92
2,0.6399,0.001,1.84
3,0.5773,0.001,2.76
4,0.5094,0.001,3.68
5,0.4603,0.001,4.6


In [35]:
ctx = ctx.with_columns(pl.Series("lisa", eda.lisa(y_tasa, celdas_m, config.PERMUTACIONES, config.SEMILLA)))
conteo_lisa = dict(ctx["lisa"].value_counts().iter_rows())
M.update({f"lisa_{k}": int(conteo_lisa.get(k, 0)) for k in ["HH", "LL", "HL", "LH"]})
print({k: M[k] for k in ("lisa_HH", "lisa_LL", "lisa_HL", "lisa_LH")})

{'lisa_HH': 299, 'lisa_LL': 239, 'lisa_HL': 39, 'lisa_LH': 3}


In [36]:
COLORES_LISA = {"HH": NARANJA, "LL": AZUL, "HL": AMARILLO, "LH": AQUA, "no_significativo": "#e1e0d9", "sin_vecinas": "#ffffff"}
g = gpd.GeoDataFrame(ctx.select("h3_cell", "lisa").to_pandas(), geometry=prep.cell_polygons(celdas_m), crs="EPSG:4326")
fig, ax = plt.subplots(figsize=(9, 8))
for tipo, color in COLORES_LISA.items():
    sub = g[g["lisa"] == tipo]
    if len(sub):
        sub.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2, label=f"{tipo} ({len(sub)})")
gpd.GeoSeries([AREA], crs="EPSG:4326").boundary.plot(ax=ax, color=TINTA, lw=1)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in COLORES_LISA.values()]
ax.legend(handles, [f"{k} ({(g['lisa'] == k).sum()})" for k in COLORES_LISA], frameon=False, loc="upper right")
ax.set(xlabel="longitud", ylabel="latitud", title=f"Clusters LISA de log(1 + tasa), I de Moran k=1 = {M['moran_I_k1']}")
guardar_figura(fig, "lisa", FASE)

PosixPath('/home/robvox/Desktop/trufi-data-science/resultados/01_datos/figuras/lisa.png')

## 11 · Diccionario de datos
**Objetivo:** documentar las columnas de las dos tablas que usa la preparación. **Salida:** `diccionario_datos.csv`.

In [37]:
DESCRIPCION = {
    "date": "fecha y hora de la consulta (texto original)", "ts": "fecha y hora parseada", "userID": "instalación de la app",
    "lat_orig": "latitud de origen", "lon_orig": "longitud de origen", "lat_dest": "latitud de destino",
    "lon_dest": "longitud de destino", "distancia": "distancia origen–destino en metros",
    "origin_municipio": "municipio de origen (geocodificación de Trufi)", "dest_municipio": "municipio de destino",
    "hour": "hora local", "day_of_week": "día de la semana (0 = lunes)", "day_of_month": "día del mes",
    "weekend": "1 si es fin de semana", "year_week_number": "semana del año (solo lote 2024)",
    "time_of_day": "franja horaria (solo lote 2024)", "year": "año ISO", "week": "semana ISO",
    "source_file": "CSV de origen", "source_batch": "lote de exportación", "source_encoding": "codificación del CSV",
    "h3_cell": "celda H3 res 8", "query_count": "consultas con origen en la celda (objetivo)",
    "user_count": "usuarios distintos con origen en la celda", "population": "población Kontur 2023",
    "population_2022": "población Kontur 2022", "lat": "latitud del centroide", "lon": "longitud del centroide",
}

In [38]:
def dicc(df: pl.DataFrame, tabla: str) -> list[dict]:
    return [{"tabla": tabla, "columna": c, "tipo": str(t), "pct_nulos": round(df[c].null_count() / df.height * 100, 3),
             "descripcion": DESCRIPCION.get(c, "")} for c, t in df.schema.items()]

diccionario = pl.DataFrame(dicc(consultas, "queries.parquet") + dicc(celdas, "celdas_objetivo.parquet"))
assert diccionario.filter(pl.col("descripcion") == "").height == 0, "Hay columnas sin descripción"
guardar_tabla(diccionario, "diccionario_datos", FASE)

tabla,columna,tipo,pct_nulos,descripcion
str,str,str,f64,str
"""queries.parquet""","""date""","""String""",0.0,"""fecha y hora de la consulta (t…"
"""queries.parquet""","""lat_orig""","""Float64""",0.0,"""latitud de origen"""
"""queries.parquet""","""lon_orig""","""Float64""",0.0,"""longitud de origen"""
"""queries.parquet""","""lat_dest""","""Float64""",0.0,"""latitud de destino"""
"""queries.parquet""","""lon_dest""","""Float64""",0.0,"""longitud de destino"""
…,…,…,…,…
"""celdas_objetivo.parquet""","""user_count""","""Int64""",0.0,"""usuarios distintos con origen …"
"""celdas_objetivo.parquet""","""population""","""Int64""",0.0,"""población Kontur 2023"""
"""celdas_objetivo.parquet""","""population_2022""","""Int64""",0.0,"""población Kontur 2022"""


## 12 · Cierre
**Objetivo:** verificar la coherencia de lo construido y guardar las métricas de la fase en `metricas.json`,
que es la única fuente de las cifras citadas en `EXPLANATIONS.md`.

In [39]:
# Coherencia: el flujo de limpieza cuadra, el objetivo conserva todas las consultas válidas y no hay nulos
assert flujo["filas_despues"][0] == N_CRUDAS and flujo["filas_despues"][-1] == limpias.height
assert (flujo["filas_eliminadas"].sum() + limpias.height) == N_CRUDAS
assert celdas["query_count"].sum() == limpias.height and (celdas["population"] >= 0).all()
assert limpias.filter(pl.col("h3_origin").is_null()).height == 0, "Consultas válidas sin celda"
assert M["semanas_observadas"] + M["semanas_faltantes"] == M["semanas_calendario"]
assert M["semanas_completas"] + M["semanas_parciales"] == M["semanas_observadas"]
assert not limpias["userID"].is_in(anomalos["userID"].implode()).any(), "Quedan usuarios anómalos"
print("Verificaciones superadas")

Verificaciones superadas


In [40]:
# Tamaño medio de la unidad (tabla H3 v4) para describir zona y macrozona
for r in (config.H3_RES, config.H3_RES_BLOQUE):
    M[f"h3_res{r}_area_km2"] = round(h3.average_hexagon_area(r, "km^2"), 3)
    M[f"h3_res{r}_arista_km"] = round(h3.average_hexagon_edge_length(r, "km"), 3)
M.update({"celdas_objetivo": celdas.height, "semilla": config.SEMILLA, "dist_max_m": config.DIST_MAX_M,
          "buffer_area_m": config.BUFFER_AREA_M, "pop_min": config.POP_MIN, "cobertura_m": config.COBERTURA_M})
ruta = guardar_metricas(FASE, M)
print(f"{len(M)} métricas → {ruta.relative_to(config.PROJECT_ROOT)}")

85 métricas → resultados/01_datos/metricas.json
